# **SN_AI – Predicting Serie A Match Results Using RNNs**

---

## Introduction / Context
The **SN_AI** project aims to predict Serie A match results from the 2015/2016 season to the ongoing 2025/2026 season.  
We will use **Recurrent Neural Networks (RNNs)** to capture temporal dependencies in historical team data and generate probabilistic match outcome predictions.

**Note:** Match results depend on more than historical results. Real-life factors such as player fitness, absences, strategies, and transfer market dynamics cannot currently be modeled due to insufficient data.

---

## Dataset and Split
- **Dataset:** each match is represented as a JSON object containing date, teams, score, and match number.  

```json
{
  "MatchNumber": 1,
  "RoundNumber": 1,
  "DateUtc": "2017-08-19 16:00:00Z",
  "Location": "Juventus Stadium",
  "HomeTeam": "Juventus",
  "AwayTeam": "Cagliari",
  "Group": null,
  "HomeTeamScore": 3,
  "AwayTeamScore": 0
}
```

**The dataset is divided in this way:**
- 70% for training
- 15% for validation
- 15% for testing

## Features
Main features used for the RNN:

- **Teams:** `home_team_name`, `away_team_name`  
- **Recent performance (last 5 matches):**  
  `home_last5_points`, `away_last5_points`  
  `home_last5_goals`, `away_last5_goals`  
- **Season averages:**  
  `home_avg_goals_scored`, `away_avg_goals_scored`  
  `home_avg_goals_conceded`, `away_avg_goals_conceded`  
- **Elo ratings:**  
  `elo_home_team`, `elo_away_team`, `elo_diff`  
- **Recent performance differences:**  
  `goal_diff_last5`, `points_diff_last5`  
- **Match history last 2 seasons:**  
  `last2yrs_match_history`  

---

## RNN Output
The network should output a Python object:

```python
result_predicted = {
    "home_win_prob": ...,
    "away_win_prob": ...,
    "draw_prob": ...,
    "home_goal_prediction": ...,
    "away_goal_prediction": ...,
    "over_2_5_prob": ...,
    "under_2_5_prob": ...
}
```

## Architecture and Method
- Network type: **RNN** (to capture temporal dependencies)  
- Activation function: **ReLU**  
- Loss function: **cross-entropy** (for result probabilities) and **MSE** (for goals)  
- Backpropagation: **BPTT (Backpropagation Through Time)**  
- Optimizer: **Adam**  
- Regularization: **dropout and L2 weight decay**  
- Validation: train/validation/test as indicated

---

## Evaluation
- Metrics:
  - Accuracy / F1-score / Precision / Recall (for win/draw/loss probabilities)  
  - MAE / MSE (for predicted goals)  
  - Confusion matrix  
- Feature importance analysis to understand which features most influence predictions

---

In [74]:
from pathlib import Path

def read_file(file_path: Path) -> str:
    """Read and return the content of a file."""
    print(f"Reading file: {file_path}")
    return file_path.read_text(encoding="utf-8")


def read_folder_files(folder_path: str) -> dict:
    """Read all files in a folder and return a dict {filename: content}."""
    folder = Path(folder_path)

    return {
        file.name: read_file(file)
        for file in folder.iterdir()
        if file.is_file()
    }


train_path = "./json_files/train"
eval_path = "./json_files/eval"
testing_path = "./json_files/testing"

train_file_contents = read_folder_files(train_path)
eval_file_contents = read_folder_files(eval_path)
testing_file_contents = read_folder_files(testing_path)

Reading file: json_files/train/SerieA_1516.json
Reading file: json_files/train/SerieA_1920.json
Reading file: json_files/train/SerieA_1819.json
Reading file: json_files/train/SerieA_2021.json
Reading file: json_files/train/SerieA_1617.json
Reading file: json_files/train/SerieA_2122.json
Reading file: json_files/train/SerieA_1718.json
Reading file: json_files/eval/SerieA_2223.json
Reading file: json_files/eval/SerieA_2324.json
Reading file: json_files/testing/SerieA_2526.json
Reading file: json_files/testing/SerieA_2425.json


In [75]:
import json
from collections import defaultdict
from enum import IntEnum
from datetime import datetime

def create_team_enum(teams_dict):
    enum_dict = {
        team.upper().replace(" ", "_"): i
        for i, team in enumerate(teams_dict.keys())
    }

    return IntEnum("SerieATeams", enum_dict)

def attach_enum_to_teams(teams_dict, enum_cls):
    return {
        team: (enum_cls[team.upper().replace(" ", "_")], points)
        for team, points in teams_dict.items()
    }

def clean_match(match: dict) -> dict:
    """Remove unnecessary keys from match."""
    keys_to_remove = {"Location", "Group"}
    return {k: v for k, v in match.items() if k not in keys_to_remove}


def update_team_points(match: dict, teams: dict):
    """Update team points based on match result."""
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    teams.setdefault(home, 0)
    teams.setdefault(away, 0)
    
    if home_score > away_score:
        teams[home] += 3
    elif home_score < away_score:
        teams[away] += 3
    else:
        teams[home] += 1
        teams[away] += 1



def track_past_match(match: dict, past_matches: dict):
    """
    Store past match results with matchDay in milliseconds since epoch.
    """
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    key = f"{home}-{away}"
    result = f"{home_score}-{away_score}"
    match_day_str = match["DateUtc"]  # esempio: "2015-08-22 18:00:00Z"

    # Converti la stringa in datetime (senza T)
    match_dt = datetime.strptime(match_day_str, "%Y-%m-%d %H:%M:%SZ")

    # Converti in millisecondi dall'epoca
    match_day_m = int(match_dt.timestamp()) / 60 # in minuti

    # Inizializza lista se non esiste
    past_matches.setdefault(key, [])
    past_matches[key].append({"result": result, "matchDay": match_day_m})

def parse_json(json_content: str, teams: dict, past_matches: dict):
    """Parse JSON content and update statistics."""
    try:
        matches = json.loads(json_content)

        if not isinstance(matches, list):
            print("Expected a list of matches")
            return []

        cleaned_matches = []

        for match in matches:
            match = clean_match(match)

            if match["HomeTeamScore"] is not None and match["AwayTeamScore"] is not None:
                update_team_points(match, teams)
                track_past_match(match, past_matches)

            cleaned_matches.append(match)

        return cleaned_matches

    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return []


def parse_dataset(file_contents: dict, teams: dict, past_matches: dict):
    """Parse all files in a dataset."""
    parsed_contents = {}

    for file, content in file_contents.items():
        parsed_contents[file] = parse_json(content, teams, past_matches)

    return parsed_contents


# Structures
all_possible_past_matches = defaultdict(list)
all_possible_teams = {}

# Parse datasets
train_parsed_contents = parse_dataset(
    train_file_contents, all_possible_teams, all_possible_past_matches
)

eval_parsed_contents = parse_dataset(
    eval_file_contents, all_possible_teams, all_possible_past_matches
)

testing_parsed_contents = parse_dataset(
    testing_file_contents, all_possible_teams, all_possible_past_matches
)

# Print results
# print("All possible past matches:")
# for match_key, match_list in all_possible_past_matches.items():
#     print(f"{match_key}: {match_list}")

SerieATeams = create_team_enum(all_possible_teams)
all_possible_teams = attach_enum_to_teams(all_possible_teams, SerieATeams)

print("\nAll possible teams with enum values and points:")
for team, (enum_value, points) in all_possible_teams.items():
    print(f"{team}: Enum={enum_value}, Points={points}")


All possible teams with enum values and points:
Hellas Verona: Enum=0, Points=315
Roma: Enum=1, Points=751
Lazio: Enum=2, Points=693
Bologna: Enum=3, Points=511
Juventus: Enum=4, Points=855
Udinese: Enum=5, Points=464
Empoli: Enum=6, Points=262
Chievoverona: Enum=7, Points=153
Fiorentina: Enum=8, Points=575
Milan: Enum=9, Points=752
Frosinone: Enum=10, Points=56
Torino: Enum=11, Points=510
Inter: Enum=12, Points=837
Atalanta: Enum=13, Points=719
Palermo: Enum=14, Points=65
Genoa: Enum=15, Points=389
Sampdoria: Enum=16, Points=354
Carpi: Enum=17, Points=38
Sassuolo: Enum=18, Points=450
Napoli: Enum=19, Points=881
Parma: Enum=20, Points=187
Crotone: Enum=21, Points=115
Cagliari: Enum=22, Points=333
Benevento: Enum=23, Points=87
Spezia: Enum=24, Points=145
Spal: Enum=25, Points=80
Pescara: Enum=26, Points=15
Salernitana: Enum=27, Points=73
Venezia: Enum=28, Points=81
Lecce: Enum=29, Points=131
Monza: Enum=30, Points=90
Cremonese: Enum=31, Points=51
Como: Enum=32, Points=149
Pisa: Enum=33

In [76]:
# Feature engineering and model training would go here, using the parsed data.
for match_key, results in all_possible_past_matches.items():
    football_match = {
        "home_team_name": all_possible_teams[match_key.split("-")[0]][0],
        "away_team_name": all_possible_teams[match_key.split("-")[1]][0],
        "home_last5_points": 10,
        "away_last5_points": 8,
        "home_last5_goals": 12,
        "away_last5_goals": 9,
        "home_avg_goals_scored": 2.4,
        "away_avg_goals_scored": 1.8,
        "home_avg_goals_conceded": 1.2,
        "away_avg_goals_conceded": 1.5,
        "elo_home_team": 1500,
        "elo_away_team": 1450,
        "elo_diff": 50,
        "goal_diff_last5": 3,
        "points_diff_last5": 2,
        "last2yrs_match_history": [
            {"season": "2022/2023", "home": "Team A", "away": "Team B", "home_score": 2, "away_score": 1},
            {"season": "2021/2022", "home": "Team B", "away": "Team A", "home_score": 0, "away_score": 0},
        ]
    }

    features = [
        football_match["home_team_name"],
        football_match["away_team_name"],
        football_match["home_last5_points"],
        football_match["away_last5_points"],
        football_match["home_avg_goals_scored"],
        football_match["away_avg_goals_scored"],
        football_match["home_avg_goals_conceded"],
        football_match["away_avg_goals_conceded"],
        football_match["elo_diff"],
        football_match["goal_diff_last5"],
        football_match["points_diff_last5"]
    ]

    print("Extracted features for the match:")
    print(features)

Extracted features for the match:
[<SerieATeams.HELLAS_VERONA: 0>, <SerieATeams.ROMA: 1>, 10, 8, 2.4, 1.8, 1.2, 1.5, 50, 3, 2]
Extracted features for the match:
[<SerieATeams.LAZIO: 2>, <SerieATeams.BOLOGNA: 3>, 10, 8, 2.4, 1.8, 1.2, 1.5, 50, 3, 2]
Extracted features for the match:
[<SerieATeams.JUVENTUS: 4>, <SerieATeams.UDINESE: 5>, 10, 8, 2.4, 1.8, 1.2, 1.5, 50, 3, 2]
Extracted features for the match:
[<SerieATeams.EMPOLI: 6>, <SerieATeams.CHIEVOVERONA: 7>, 10, 8, 2.4, 1.8, 1.2, 1.5, 50, 3, 2]
Extracted features for the match:
[<SerieATeams.FIORENTINA: 8>, <SerieATeams.MILAN: 9>, 10, 8, 2.4, 1.8, 1.2, 1.5, 50, 3, 2]
Extracted features for the match:
[<SerieATeams.FROSINONE: 10>, <SerieATeams.TORINO: 11>, 10, 8, 2.4, 1.8, 1.2, 1.5, 50, 3, 2]
Extracted features for the match:
[<SerieATeams.INTER: 12>, <SerieATeams.ATALANTA: 13>, 10, 8, 2.4, 1.8, 1.2, 1.5, 50, 3, 2]
Extracted features for the match:
[<SerieATeams.PALERMO: 14>, <SerieATeams.GENOA: 15>, 10, 8, 2.4, 1.8, 1.2, 1.5, 50, 3, 